- Daily Sales 
- Monthly Sales 
- Top Products 
- Top Customers 

### Creating Main Table(fact order)

In [0]:
from delta.tables import DeltaTable

# Source DataFrame
df_silver_order = spark.table("associate_assignment.default.silver_orders")

# Target Delta Table
target_table = DeltaTable.forName(
    spark,
    "associate_assignment.default.fact_main_order"
)

(
    target_table.alias("t")
    .merge(
        source=df_silver_order.alias("s"),
        condition="t.order_id = s.order_id"
    )
    .whenMatchedUpdate(
        set={
            "customer_id": "s.customer_id",
            "product_id": "s.product_id",
            "order_date": "s.order_date",
            "quantity": "s.quantity",
            "price": "s.price",
            "total_amount": "s.total_amount",
            "source_file": "s.source_file",
            "ingestion_timestamp": "s.ingestion_timestamp"
        }
    )
    .whenNotMatchedInsert(
        values={
            "order_id": "s.order_id",
            "customer_id": "s.customer_id",
            "product_id": "s.product_id",
            "order_date": "s.order_date",
            "quantity": "s.quantity",
            "price": "s.price",
            "total_amount": "s.total_amount",
            "source_file": "s.source_file",
            "ingestion_timestamp": "s.ingestion_timestamp"
        }
    )
    .execute()
)

### DAILY SALES

In [0]:
df_daily_sales = spark.sql("""SELECT order_date,SUM(total_amount) AS DAILY_SALES FROM associate_assignment.default.fact_main_order 
GROUP BY order_date
ORDER BY order_date""")

df_daily_sales.write.mode("overwrite").saveAsTable("associate_assignment.default.dim_daily_sales")

### Monthly Sales

In [0]:
df_monthly_sales = spark.sql(""" SELECT
    YEAR(order_date) AS year,
    MONTH(order_date) AS month,
    SUM(total_amount) AS monthly_sales
FROM associate_assignment.default.fact_main_order
GROUP BY
    YEAR(order_date),
    MONTH(order_date)
ORDER BY
    year,
    month;
""")

df_monthly_sales.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("associate_assignment.default.dim_monthly_sales")

### Top Costomers

In [0]:
df_top_customer = spark.sql("""select customer_id,sum(total_amount) as total_sales from associate_assignment.default.fact_main_order
group by customer_id
order by total_sales desc
limit 10""")

df_top_customer.write.mode("overwrite").saveAsTable("associate_assignment.default.dim_top_customer")

### Top Products

In [0]:
df_top_product = spark.sql("""
SELECT
    product_id,
    SUM(total_amount) AS total_sales
FROM associate_assignment.default.fact_main_order
GROUP BY product_id
ORDER BY total_sales DESC
LIMIT 10
""")

df_top_product.write.mode("overwrite").saveAsTable("associate_assignment.default.dim_top_products")